In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import re
import os
import sys

sys.path.append(os.path.abspath('../'))

In [2]:
raw_path = "../data/raw/"
processed_path = "../data/processed/"
figures_path= "../reports/figures/"
src_path = "../src/"

In [3]:
# Load dữ liệu 
df_listings = pd.read_csv(os.path.join(raw_path, 'listings.csv'))
df_calendar = pd.read_csv(os.path.join(raw_path, 'calendar_2025.csv'))                  

In [4]:
df_calendar.info()
df_listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 495670 entries, 0 to 495669
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   listing_id      495670 non-null  int64  
 1   date            495670 non-null  object 
 2   available       495670 non-null  object 
 3   price           0 non-null       float64
 4   adjusted_price  0 non-null       float64
 5   minimum_nights  495670 non-null  int64  
 6   maximum_nights  495670 non-null  int64  
dtypes: float64(2), int64(3), object(2)
memory usage: 26.5+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1358 entries, 0 to 1357
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            1358 non-null   int64  
 1   listing_url                                   1358 non-null   object 
 2   s

## PHẦN 1: PHÂN TÍCH SƠ BỘ TẬP DỮ LIỆU CALENDAR_2025

Tập dữ liệu calendar phản ánh trạng thái đặt phòng theo từng ngày, giúp nắm bắt hành vi thực tế của thị trường.

Mục tiêu của bước này là xây dựng biến occupancy_rate nhằm đại diện cho mức độ sức hút thị trường của từng căn hộ.

In [5]:
df_calendar.head()

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,8521,2025-09-28,f,NaN,NaN,2,1125
1,8521,2025-09-29,f,NaN,NaN,2,1125
2,8521,2025-09-30,f,NaN,NaN,2,1125
3,8521,2025-10-01,f,NaN,NaN,2,1125
4,8521,2025-10-02,f,NaN,NaN,2,1125


In [6]:
df_calendar.describe()

,listing_id,price,adjusted_price,minimum_nights,maximum_nights
count,4.956700e+05,0.0,0.0,495670.000000,495670.000000
mean,6.303908e+17,NaN,NaN,22.519142,639.369244
std,5.666651e+17,NaN,NaN,36.787509,519.096546
min,8.521000e+03,NaN,NaN,1.000000,1.000000
25%,3.405758e+07,NaN,NaN,2.000000,365.000000
50%,6.827960e+17,NaN,NaN,15.000000,365.000000
75%,1.178492e+18,NaN,NaN,30.000000,1125.000000
max,1.517698e+18,NaN,NaN,700.000000,11250.000000


In [7]:
df_calendar.isnull().sum()  

listing_id             0
date                   0
available              0
price             495670
adjusted_price    495670
minimum_nights         0
maximum_nights         0
dtype: int64

Sau khi khám phá tổng quan tập dữ liệu calendar, chúng ta nhận thấy tập dữ liệu gồm 495.670 bản ghi với 7 trường thông tin.

Một vấn đề đáng chú ý là hai cột `price` và `adjusted_price` bị khuyết thiếu hoàn toàn (100% giá trị null). Điều này cho thấy hai biến này không chứa bất kỳ thông tin hữu ích nào cho phân tích (information gain = 0).

Việc giữ lại các cột này không chỉ làm tăng độ phức tạp không cần thiết của dữ liệu mà còn có thể gây nhiễu trong các bước xử lý tiếp theo, đặc biệt khi thực hiện các phép thống kê hoặc mô hình hóa. Việc loại bỏ các cột không mang thông tin cũng giúp cải thiện hiệu suất tính toán và giảm nguy cơ phát sinh lỗi trong quá trình xử lý dữ liệu.

Do đó, nhóm quyết định loại bỏ hoàn toàn hai cột này nhằm đảm bảo tập dữ liệu gọn gàng, nhất quán và tối ưu cho các bước phân tích tiếp theo.

In [8]:
df_calendar.dropna(axis=1, how='all', inplace=True)

In [9]:
df_calendar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 495670 entries, 0 to 495669
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   listing_id      495670 non-null  int64 
 1   date            495670 non-null  object
 2   available       495670 non-null  object
 3   minimum_nights  495670 non-null  int64 
 4   maximum_nights  495670 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 18.9+ MB


Sau khi loại bỏ các cột trống hoàn toàn, chúng ta Bước tiếp theo, để đảm bảo tính toàn vẹn của chuỗi thời gian, chúng ta cần tiến hành rà soát các dòng dữ liệu ghi nhận dữ liệu trùng lặp ngày

In [10]:
# Kiểm tra dữ liệu trùng lặp
df_calendar.duplicated().sum()
df_calendar.duplicated(subset=['listing_id', 'date']).sum()

np.int64(0)

Kết quả kiểm tra cho thấy tập dữ liệu Calendar đảm bảo tính vẹn toàn dữ liệu (không có dữ liệu lặp). Dữ liệu đã sạch giá trị nhiễu và sẵn sàng cho bước chuyển đổi đặc trưng. Mô hình học máy không thể tính toán dữ liệu dạng chuỗi của cột available. Do đó, ta cần số hóa cột này: gán giá trị 1 cho những ngày đã được đặt (f) và 0 cho những ngày còn trống (t).

In [11]:
df_calendar['available'].isnull().sum()

np.int64(0)

In [12]:
df_calendar['available'].value_counts()

available
f    249564
t    246106
Name: count, dtype: int64

In [13]:
from src.feature_engineering import encode_binary_features
binary_cols = ['available']
df_calendar = encode_binary_features(df_calendar, binary_cols)

In [14]:
df_calendar['available'].head(10)

0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
9    0
Name: available, dtype: int64

Sau khi tiến hành binarization, chúng ta tiến hành gom nhóm theo từng căn hộ (listing_id) để tính tỷ lệ lấp đày trung bình trong năm. Hành động này giúp nén dữ liệu thời gian thành bảng dữ liệu chéo, chuẩn bị cho việc hợp nhất dữ liệu với tập listings.

In [15]:
df_occupancy = df_calendar.groupby('listing_id')['available'].mean().reset_index()

In [16]:
df_occupancy.rename(columns={'available': 'occupancy_rate'}, inplace=True)
df_occupancy.head()

,listing_id,occupancy_rate
0,8521,0.257534
1,11169,0.852055
2,19581,0.805479
3,27498,0.934247
4,79762,0.824658


In [17]:
df_occupancy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1358 entries, 0 to 1357
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   listing_id      1358 non-null   int64  
 1   occupancy_rate  1358 non-null   float64
dtypes: float64(1), int64(1)
memory usage: 21.3 KB


## KIỂM TRA TÍNH TOÀN VẸN DỮ LIỆU (DATA INTEGRITY)

Sau khi thực hiện nén dữ liệu từ tập *calendar_2025* về cấp độ từng căn hộ (listing_id), nhóm tiến hành đối chiếu với tập *listings* nhằm xác nhận tính nhất quán của dữ liệu.

In [18]:
calendar_ids = set(df_occupancy['listing_id'])
listing_ids = set(df_listings['id'])

if calendar_ids == listing_ids:
    print("Dữ liệu khớp hoàn toàn 100%")
else:
    print(f"Có {len(calendar_ids - listing_ids)} ID không khớp.")

Dữ liệu khớp hoàn toàn 100%


Kết quả cho thấy 100% mã định danh (1.358 listing_id) giữa hai tập dữ liệu trùng khớp hoàn toàn. Điều này chứng minh rằng quá trình xử lý không làm mất mát bản ghi cũng như không phát sinh các ID bất thường.

Nhờ đó, chúng ta có thể tự tin thực hiện bước hợp nhất dữ liệu (merge) mà không lo ngại về sai lệch hoặc thiếu hụt dữ liệu.

## GIÁ TRỊ CỦA BIẾN OCCUPANCY_RATE

Thông qua kỹ thuật gom nhóm (groupby), gần 500.000 dòng dữ liệu giao dịch theo ngày đã được chuyển đổi thành một biến đặc trưng duy nhất cho mỗi căn hộ.

Biến occupancy_rate không chỉ đơn thuần phản ánh tỷ lệ lấp đầy, mà còn đại diện cho mức độ hấp dẫn thực tế của căn hộ trên thị trường.

Trong bối cảnh bài toán định giá, chỉ dựa vào giá niêm yết có thể dẫn đến hiện tượng "giá ảo" — khi giá cao nhưng không có nhu cầu tương ứng.

Ngược lại, occupancy_rate cung cấp một tín hiệu về hành vi thực tế của khách hàng, giúp mô hình phân biệt giữa giá niêm yết và giá trị thị trường thực sự.

Do đó, việc đưa biến này vào mô hình giúp cải thiện đáng kể độ chính xác và tính thực tiễn của bài toán định giá.

## LƯU DỮ LIỆU SAU XỬ LÝ

Sau khi hoàn tất quá trình xử lý và kiểm tra tính toàn vẹn dữ liệu, tập dữ liệu df_occupancy được lưu lại để phục vụ cho các bước phân tích tiếp theo.

Việc lưu trữ dữ liệu trung gian giúp đảm bảo tính tái sử dụng và khả năng kiểm soát pipeline dữ liệu.

In [19]:
df_occupancy.to_csv('../data/processed/df_occupancy.csv', index=False)

## PHẦN 2: KHÁM PHÁ ĐẶC TRƯNG CĂN HỘ (EDA - LISTINGS)

Sau khi xác nhận tính đồng nhất về mã định danh (id) và xây dựng thành công biến số chiến lược occupancy_rate từ dữ liệu thời gian (calendar), chúng ta chuyển sang phân tích tập dữ liệu chính: **listings.csv**.

Nếu occupancy_rate đại diện cho "nhu cầu thị trường" (demand), thì tập listings phản ánh "nguồn cung" (supply) — bao gồm các đặc điểm cấu thành giá trị của từng căn hộ tại Cambridge.

Tập dữ liệu này đóng vai trò là "khung xương" của mô hình, cung cấp các thông tin về đặc điểm vật lý, vị trí và chính sách vận hành của từng căn hộ.

Việc thực hiện EDA cho tập dữ liệu này nhằm phục vụ trực tiếp cho bài toán định giá, với 3 mục tiêu trọng tâm:

1. **Tinh lọc đặc trưng (Feature Selection)**  
Loại bỏ các trường dữ liệu nhiễu (URL, mô tả văn bản dài, ID người dùng) để tập trung vào các biến có khả năng ảnh hưởng đến giá và hiệu suất kinh doanh như: room_type, accommodates, bathrooms, bedrooms, và neighbourhood.

2. **Làm sạch dữ liệu (Data Cleaning)**  
Chuẩn hóa kiểu dữ liệu (đặc biệt là cột price), xử lý giá trị thiếu (Missing Values) và kiểm soát các điểm dị biệt (Outliers) nhằm đảm bảo tính ổn định cho các phân tích và mô hình sau này.

3. **Hiểu cấu trúc thị trường (Distribution Analysis)**  
Phân tích phân phối của các biến quan trọng để nắm bắt bức tranh tổng quan về thị trường Airbnb tại Cambridge, bao gồm:
- Sự phân bố loại hình căn hộ  
- Mức giá phổ biến  
- Sự khác biệt giữa các khu vực  

Bước này đóng vai trò nền tảng trước khi hợp nhất dữ liệu với occupancy_rate, từ đó cho phép phân tích mối quan hệ giữa giá, đặc điểm căn hộ và hành vi thị trường.

In [20]:
# kiểm tra tập dữ liệu listings
df_listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1358 entries, 0 to 1357
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            1358 non-null   int64  
 1   listing_url                                   1358 non-null   object 
 2   scrape_id                                     1358 non-null   int64  
 3   last_scraped                                  1358 non-null   object 
 4   source                                        1358 non-null   object 
 5   name                                          1358 non-null   object 
 6   description                                   1347 non-null   object 
 7   neighborhood_overview                         694 non-null    object 
 8   picture_url                                   1358 non-null   object 
 9   host_id                                       1358 non-null   i

In [21]:
df_listings.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,8521,https://www.airbnb.com/rooms/8521,20250928035003,2025-09-28,city scrape,SunsplashedSerenity walk to Harvard & Fresh Pond,"An elegant, sun-splashed, 2 bedroom (+2offices...",Huron Village is known for its charm. We have...,https://a0.muscache.com/pictures/30536/072e0a5...,306681,...,4.92,4.94,4.74,C0121120491,f,2,2,0,0,0.41
1,11169,https://www.airbnb.com/rooms/11169,20250928035003,2025-09-28,city scrape,Lovely Studio Room: Available for long w/ends,Large sunny room w kitchenette & bath. Foam ma...,The neighborhood is quiet and friendly and our...,https://a0.muscache.com/pictures/miso/Hosting-...,40965,...,4.92,4.80,4.77,NaN,f,3,1,2,0,1.01
2,19581,https://www.airbnb.com/rooms/19581,20250928035003,2025-09-28,city scrape,"Furnished suite, Windsor","Welcome to Area IV! We are located, convenient...",NaN,https://a0.muscache.com/pictures/188f1b4b-f37b...,74249,...,4.91,4.91,4.27,NaN,t,3,0,3,0,0.07
3,27498,https://www.airbnb.com/rooms/27498,20250928035003,2025-09-28,city scrape,Furnished suite 2 @ the Windsor,"Welcome to Area IV! We are located, convenient...",NaN,https://a0.muscache.com/pictures/bab30c3c-ff3c...,74249,...,4.76,4.88,4.64,NaN,t,3,0,3,0,0.15
4,79762,https://www.airbnb.com/rooms/79762,20250928035003,2025-09-28,city scrape,Cambridge Getaway @ Harvard & MIT,Charming 2-bedroom apartment on the third floo...,Annmarie and I have lived in this area for ove...,https://a0.muscache.com/pictures/airflow/Hosti...,430015,...,4.93,4.93,4.77,STR-15661,f,1,1,0,0,2.59


In [22]:
df_listings.describe()

,id,scrape_id,host_id,host_listings_count,host_total_listings_count,neighbourhood_group_cleansed,latitude,longitude,accommodates,bathrooms,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
count,1.358000e+03,1.358000e+03,1.358000e+03,1358.000000,1358.000000,0.0,1358.000000,1358.000000,1358.000000,1065.000000,...,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1358.000000,1358.000000,1358.000000,1358.000000,1025.000000
mean,6.303908e+17,2.025093e+13,1.777369e+08,698.770250,851.924153,NaN,42.373912,-71.106471,3.270250,1.307981,...,4.744244,4.870020,4.838771,4.872527,4.616556,38.865979,27.310015,10.977172,0.000736,1.525834
std,5.668733e+17,0.000000e+00,1.949916e+08,1681.470771,1939.485396,NaN,0.010625,0.019023,2.315851,0.595237,...,0.417628,0.303552,0.380003,0.281349,0.480144,56.951799,54.644200,20.183577,0.027136,1.909016
min,8.521000e+03,2.025093e+13,3.538400e+04,1.000000,1.000000,NaN,42.353670,-71.157580,1.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.010000
25%,3.414261e+07,2.025093e+13,2.436748e+07,3.000000,4.000000,NaN,42.366240,-71.119623,2.000000,1.000000,...,4.670000,4.860000,4.830000,4.860000,4.530000,2.000000,0.000000,0.000000,0.000000,0.240000
50%,6.827960e+17,2.025093e+13,1.074344e+08,16.000000,24.000000,NaN,42.370599,-71.105840,2.000000,1.000000,...,4.880000,4.950000,4.950000,4.930000,4.720000,8.000000,2.000000,1.000000,0.000000,0.850000
75%,1.178137e+18,2.025093e+13,3.736751e+08,169.000000,234.000000,NaN,42.381308,-71.091340,4.000000,1.500000,...,4.990000,5.000000,5.000000,5.000000,4.860000,83.000000,26.000000,12.000000,0.000000,2.250000
max,1.517698e+18,2.025093e+13,7.207319e+08,5143.000000,5933.000000,NaN,42.400180,-71.071239,16.000000,4.000000,...,5.000000,5.000000,5.000000,5.000000,5.000000,169.000000,169.000000,71.000000,1.000000,24.220000


In [23]:
missing_data=df_listings.isnull().sum().sort_values(ascending=False)
print(missing_data[missing_data > 0])

neighbourhood_group_cleansed    1358
calendar_updated                1358
license                          892
neighbourhood                    664
neighborhood_overview            664
host_about                       480
review_scores_communication      333
last_review                      333
reviews_per_month                333
first_review                     333
review_scores_location           333
review_scores_accuracy           333
review_scores_rating             333
review_scores_value              333
review_scores_cleanliness        333
review_scores_checkin            333
price                            303
estimated_revenue_l365d          303
beds                             300
bathrooms                        293
host_location                    250
host_response_rate               111
host_response_time               111
bedrooms                          93
host_acceptance_rate              86
host_is_superhost                 41
host_neighbourhood                34
h

## ĐÁNH GIÁ CHẤT LƯỢNG DỮ LIỆU VÀ CHIẾN LƯỢC XỬ LÝ

Dựa trên kết quả thống kê mô tả và kiểm tra giá trị khuyết thiếu, nhóm xác định các vấn đề trọng yếu ảnh hưởng trực tiếp đến bài toán định giá Airbnb tại Cambridge, đồng thời đề xuất các phương án xử lý như sau:

**1. Kiểm soát biến mục tiêu (price)**  
Với tỷ lệ thiếu khoảng ~22%, biến price được xem là yếu tố cốt lõi trong bài toán. Nhóm ưu tiên rà soát chéo với tập calendar để xác minh. Các bản ghi không thể xác định giá niêm yết sẽ bị loại bỏ nhằm đảm bảo độ tin cậy và tính nhất quán của mô hình dự báo.

**2. Tối ưu hóa dữ liệu đặc trưng (Feature Engineering)**  
Cột bathrooms có tỷ lệ thiếu cao, trong khi bathrooms_text gần như đầy đủ. Nhóm tận dụng cột này để trích xuất thông tin số lượng phòng tắm, qua đó khôi phục dữ liệu và tăng giá trị sử dụng của biến đặc trưng.

**3. Lọc nhiễu và tinh gọn dữ liệu (Noise Reduction)**  
Các cột trống hoàn toàn và siêu dữ liệu không mang giá trị dự báo sẽ được loại bỏ. Đối với biến license, thay vì loại bỏ, nhóm thực hiện mã hóa nhị phân nhằm khai thác thông tin về tính pháp lý và độ tin cậy của căn hộ.

**4. Chiến lược xử lý giá trị thiếu (Imputation)**  
Các biến quan trọng như bedrooms, beds và review_scores được xử lý bằng giá trị trung vị (Median) nhằm hạn chế ảnh hưởng của các giá trị cực biên, đồng thời giữ nguyên phân phối dữ liệu.

Cách tiếp cận này đảm bảo dữ liệu đầu vào vừa sạch, vừa giữ được tối đa giá trị thông tin phục vụ cho bài toán định giá.

## THIẾT LẬP QUY TRÌNH XỬ LÝ DỮ LIỆU

Nhóm triển khai quy trình xử lý theo nguyên tắc: **Làm sạch trước – Biến đổi sau**, nhằm đảm bảo tính toàn vẹn và khả năng khai thác dữ liệu hiệu quả.

Quy trình bao gồm 5 bước chính:

**1. Noise Filtering**  
Loại bỏ các đặc trưng không mang thông tin (100% null) và dữ liệu nhiễu để giảm độ phức tạp của tập dữ liệu.

**2. Target Normalization**  
Chuẩn hóa biến price và loại bỏ các bản ghi không có giá trị mục tiêu, đảm bảo tính hợp lệ cho bài toán học có giám sát.

**3. Numerical Cleaning**  
Xử lý các giá trị thiếu bằng Median và kiểm soát các điểm dị biệt (Outliers) thông qua phương pháp IQR nhằm bảo toàn phân phối dữ liệu.

**4. Feature Enrichment**  
Trích xuất thông tin từ dữ liệu văn bản (bathrooms_text) và mã hóa các biến định tính quan trọng (license, superhost) thành dạng số.

**5. Data Integration**  
Hợp nhất dữ liệu đặc trưng căn hộ với biến occupancy_rate nhằm kết nối giữa "nguồn cung" và "nhu cầu thị trường", tạo nền tảng cho phân tích định giá.

Pipeline này đảm bảo dữ liệu đầu vào không chỉ sạch về mặt kỹ thuật mà còn có ý nghĩa về mặt kinh doanh, phục vụ trực tiếp cho việc tối ưu hóa giá niêm yết.

Trong tập dữ liệu gốc, trường license chứa các mã định danh pháp lý dưới dạng chuỗi ký tự (Strings). Nhóm thực hiện chuyển đổi thông tin này sang dạng nhị phân (Binary Encoding) để chuẩn bị dữ liệu cho các bước kiểm định sau này:

has_license = 1: Căn hộ có thông tin giấy phép hiện hữu.

has_license = 0: Căn hộ thiếu thông tin hoặc chưa niêm yết giấy phép.

Việc trích xuất này nhằm mục đích giữ lại dữ liệu về trạng thái pháp lý - một yếu tố được kỳ vọng là có tiềm năng tác động đến tâm lý người thuê hoặc chiến lược định giá của chủ nhà. Biến số này sẽ đóng vai trò là một "giả thuyết đầu vào" để nhóm tiến hành phân tích tương quan và kiểm chứng sức ảnh hưởng thực tế ở các giai đoạn sau của dự án.

In [24]:
df_listings.drop(
    columns=['neighbourhood_group_cleansed', 'calendar_updated'],
    inplace=True
)

In [25]:
df_listings['price']

0         $270.00
1         $126.00
2         $183.00
3         $238.00
4         $300.00
          ...    
1353      $135.00
1354    $1,166.00
1355    $1,018.00
1356    $1,166.00
1357       $36.00
Name: price, Length: 1358, dtype: object

In [26]:
from src.data_cleaning import clean_price
df_listings = clean_price(df_listings)

In [27]:
df_listings['price'].isnull().sum()

np.int64(303)

Trong quá trình khám phá dữ liệu, nhóm thực hiện kiểm tra các giá trị thiếu trên toàn bộ tập dữ liệu listings. Kết quả cho thấy trường price (biến mục tiêu $Y$) có 303 bản ghi khuyết thiếu (chiếm khoảng 22% tổng số 1.358 quan sát ban đầu). Nhóm quyết định thực hiện loại bỏ hoàn toàn các bản ghi này thay vì áp dụng các kỹ thuật xử lý missing value dựa trên các cơ sở khoa học sau: Bản chất của biến phụ thuộc ($Y$): Trong các mô hình hồi quy hoặc dự báo kinh tế, price đóng vai trò là "nhãn" (Label) để máy tính học tập. Việc tự ý điền khuyết giá trị mục tiêu bằng giá trị trung bình hay trung vị sẽ vô tình tạo ra các "dữ liệu giả" (Synthetic Data), làm sai lệch phân phối thực tế của thị trường và dẫn đến hiện tượng Overfitting (mô hình chỉ học được các con số giả định thay vì quy luật thị trường thực tế).Đảm bảo độ tin cậy của Ground Truth: Một căn hộ không niêm yết giá được coi là "quan sát không có nhãn" (Unlabeled observation). Việc giữ lại chúng sẽ làm nhiễu các chỉ số đánh giá mô hình như $R^2$ hay $RMSE$ sau này.Tối ưu hóa phễu dữ liệu: Sau khi loại bỏ 303 dòng nhiễu, tập dữ liệu còn lại 1.055 dòng "nạc" đảm bảo tính nhất quán 100% về mặt thông tin (vừa có đặc trưng đặc điểm, vừa có giá niêm yết thực tế).Hành động: Nhóm triển kha

In [28]:
df_listings.dropna(subset=['price'], inplace=True)

In [29]:
df_listings.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1055 entries, 0 to 1357
Data columns (total 77 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            1055 non-null   int64  
 1   listing_url                                   1055 non-null   object 
 2   scrape_id                                     1055 non-null   int64  
 3   last_scraped                                  1055 non-null   object 
 4   source                                        1055 non-null   object 
 5   name                                          1055 non-null   object 
 6   description                                   1045 non-null   object 
 7   neighborhood_overview                         541 non-null    object 
 8   picture_url                                   1055 non-null   object 
 9   host_id                                       1055 non-null   int64 

In [30]:
df_listings['bathrooms_text'].isnull().sum()

np.int64(1)

In [31]:
from src.data_cleaning import fill_bathrooms_text
df_listings = fill_bathrooms_text(df_listings)

In [32]:
from src.feature_engineering import add_bath_features
df_listings = add_bath_features(df_listings)

In [33]:
df_listings[['bath_qty', 'is_shared_bath']].isnull().sum()

bath_qty          0
is_shared_bath    0
dtype: int64

In [34]:
df_listings[['bath_qty', 'is_shared_bath']].head(10)

,bath_qty,is_shared_bath
0,1.0,0.0
1,1.0,0.0
2,1.0,0.0
3,1.0,0.0
4,1.0,0.0
5,1.0,0.0
6,1.0,0.0
9,1.0,1.0
10,1.0,0.0
11,1.0,0.0


In [35]:
df_listings['room_type'].value_counts()

room_type
Entire home/apt    593
Private room       436
Hotel room          26
Name: count, dtype: int64

In [36]:
from src.feature_engineering import encode_room_type
df_listings = encode_room_type(df_listings)

In [37]:
df_listings['beds'].isnull().sum()

np.int64(3)

In [38]:
from src.data_cleaning import fill_beds
df_listings = fill_beds(df_listings)

In [39]:
df_listings['beds'].isnull().sum()

np.int64(0)

In [40]:
df_listings['amenities'].isnull().sum()

np.int64(0)

In [41]:
df_listings['amenities'].value_counts().head(10)

amenities
["Pets allowed", "TV", "Air conditioning", "Paid parking on premises", "Hangers", "First aid kit", "Extra pillows and blankets", "Essentials", "Wifi", "Fire extinguisher", "Building staff", "Coffee maker", "Microwave", "Carbon monoxide alarm", "Bed linens", "Shampoo", "Hair dryer", "Iron", "Refrigerator", "Dedicated workspace", "Self check-in", "Gym", "Smoke alarm", "Pool", "Heating"]                                                                                                                                                                                                                                                                                         24
["Pets allowed", "High chair", "TV", "Air conditioning", "Hangers", "Dryer \u2013\u00a0In unit", "Lockbox", "Essentials", "Wifi", "BBQ grill", "Oven", "Coffee maker", "Microwave", "Elevator", "Cooking basics", "Carbon monoxide alarm", "Bed linens", "Shampoo", "Hot water", "Washer \u2013\u00a0In unit", "Hair dryer", "I

In [42]:
from src.data_cleaning import clean_amenities
cleaned_amenities = clean_amenities(df_listings)

In [43]:
all_amenities = cleaned_amenities.explode().dropna()
all_amenities = all_amenities[all_amenities != '']

amenity_counts = Counter(all_amenities)

import pandas as pd
df_amenity_freq = pd.DataFrame(
    amenity_counts.items(),
    columns=['Amenity', 'Count']
).sort_values(by='Count', ascending=False)

top_amenities = df_amenity_freq.head(10)['Amenity'].tolist()

print(top_amenities)

['Smoke alarm', 'Carbon monoxide alarm', 'Wifi', 'Hot water', 'Hangers', 'Essentials', 'Bed linens', 'Kitchen', 'Hair dryer', 'Refrigerator']


In [44]:
from src.feature_engineering import add_top_amenity_features
df_listings = add_top_amenity_features(df_listings, top_amenities)

In [45]:
from src.feature_engineering import add_amenities_count
df_listings = add_amenities_count(df_listings, cleaned_amenities)

In [46]:
df_listings['property_type'].value_counts()

property_type
Entire rental unit                    446
Private room in rental unit           210
Private room in home                  132
Entire home                            56
Entire condo                           52
Room in hotel                          48
Room in boutique hotel                 28
Private room in townhouse              16
Private room in bed and breakfast      13
Entire guest suite                     11
Entire townhouse                       10
Entire serviced apartment               6
Private room in condo                   6
Entire loft                             5
Private room in casa particular         4
Private room in guest suite             3
Entire villa                            2
Entire guesthouse                       2
Private room in serviced apartment      1
Private room in cottage                 1
Tiny home                               1
Boat                                    1
Entire place                            1
Name: count, dtype: 

In [47]:
from src.feature_engineering import simplify_property_type
df_listings['property_type']=df_listings['property_type'].apply(simplify_property_type)

In [48]:
df_listings['property_type'].value_counts()

property_type
Apartment        726
House            220
Hotel_B_and_B     93
Other_Unique      16
Name: count, dtype: int64

In [49]:
from src.feature_engineering import encode_property_type
df_listings = encode_property_type(df_listings, 'property_type')

Đã tạo ra 3 cột mới: ['property_type_Hotel_B_and_B', 'property_type_House', 'property_type_Other_Unique']


In [50]:
print(df_listings[[col for col in df_listings.columns if 'property_type' in col]].head())

   property_type_Hotel_B_and_B  property_type_House  \
0                            0                    0   
1                            0                    0   
2                            1                    0   
3                            1                    0   
4                            0                    0   

   property_type_Other_Unique  
0                           0  
1                           0  
2                           0  
3                           0  
4                           0  


In [51]:
df_listings['host_is_superhost'].value_counts()

host_is_superhost
f    624
t    396
Name: count, dtype: int64

In [52]:
df_listings['host_is_superhost'].isnull().sum()

np.int64(35)

In [53]:
from src.feature_engineering import encode_binary_features
binary_cols = ['host_is_superhost']
df_listings = encode_binary_features(df_listings, binary_cols)

In [54]:
df_listings['host_is_superhost'] = df_listings['host_is_superhost'].fillna(0).astype(int)

In [55]:
df_listings['host_is_superhost'].isnull().sum()

np.int64(0)

In [56]:
df_listings['host_is_superhost'].head(10)

0     1
1     1
2     1
3     1
4     1
5     1
6     1
9     0
10    1
11    0
Name: host_is_superhost, dtype: int64

In [57]:
df_listings['instant_bookable'].isnull().sum()

np.int64(0)

In [58]:
df_listings['instant_bookable'].value_counts()

instant_bookable
f    634
t    421
Name: count, dtype: int64

In [59]:
from src.feature_engineering import encode_binary_features
binary_cols = ['instant_bookable']
df_listings = encode_binary_features(df_listings, binary_cols)

In [60]:
df_listings['instant_bookable'].head(10)

0     0
1     0
2     1
3     1
4     0
5     1
6     0
9     1
10    0
11    0
Name: instant_bookable, dtype: int64

In [61]:
df_listings['license'].head(10) 

0          C0121120491
1                  NaN
2                  NaN
3                  NaN
4            STR-15661
5     ROT-10694669-006
6          CO121110491
9                  NaN
10         C0262370491
11                 NaN
Name: license, dtype: object

In [62]:
df_listings['license'].isnull().sum()

np.int64(696)

In [63]:
from src.feature_engineering import add_license_feature
df_listings = add_license_feature(df_listings)

In [64]:
df_listings['has_license'].value_counts()

has_license
0    696
1    359
Name: count, dtype: int64

In [65]:
df_listings['has_license'].isnull().sum()

np.int64(0)

In [66]:
df_listings['review_scores_accuracy'].isnull().sum()

np.int64(207)

In [67]:
df_listings['review_scores_cleanliness'].isnull().sum()

np.int64(207)

In [68]:
df_listings['review_scores_communication'].isnull().sum()

np.int64(207)

In [69]:
df_listings['review_scores_location'].isnull().sum()

np.int64(207)

In [70]:
df_listings['review_scores_value'].isnull().sum()

np.int64(207)

In [71]:
df_listings['review_scores_rating'].isnull().sum()

np.int64(207)

In [72]:
df_listings['review_scores_checkin'].isnull().sum()

np.int64(207)

In [73]:
df_listings['has_review'] = df_listings['review_scores_rating'].notnull().astype(int)


In [74]:
from src.data_cleaning import fill_review_scores
df_listings = fill_review_scores(df_listings)

In [75]:
df_listings['review_scores_checkin'].isnull().sum()

np.int64(0)

In [76]:
df_listings['review_scores_rating'].isnull().sum()

np.int64(0)

In [77]:
df_listings['review_scores_value'].isnull().sum()

np.int64(0)

In [78]:
df_listings['review_scores_location'].isnull().sum()

np.int64(0)

In [79]:
df_listings['review_scores_cleanliness'].isnull().sum()

np.int64(0)

In [80]:
df_listings['review_scores_communication'].isnull().sum()

np.int64(0)

In [81]:
df_listings['review_scores_accuracy'].isnull().sum()

np.int64(0)

In [82]:
df_listings.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1055 entries, 0 to 1357
Data columns (total 95 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            1055 non-null   int64  
 1   listing_url                                   1055 non-null   object 
 2   scrape_id                                     1055 non-null   int64  
 3   last_scraped                                  1055 non-null   object 
 4   source                                        1055 non-null   object 
 5   name                                          1055 non-null   object 
 6   description                                   1045 non-null   object 
 7   neighborhood_overview                         541 non-null    object 
 8   picture_url                                   1055 non-null   object 
 9   host_id                                       1055 non-null   int64 

In [83]:
cols_drop = [
    'scrape_id', 'last_scraped', 'source', 'calendar_last_scraped', 'listing_url', 'host_url', 'picture_url', 'host_thumbnail_url', 'host_picture_url',
    
    'host_name','host_since', 'host_id', 'host_has_profile_pic', 'host_about', 'host_location', 'host_acceptance_rate',
    'host_verifications', 'host_neighbourhood', 
    'host_response_time', 'host_response_rate', 
    'minimum_minimum_nights', 'maximum_minimum_nights',
    'host_identity_verified',
'minimum_maximum_nights', 'maximum_maximum_nights','minimum_nights_avg_ntm'
, 'maximum_nights_avg_ntm',
 'calculated_host_listings_count_entire_homes',
    'calculated_host_listings_count_private_rooms',
    'calculated_host_listings_count_shared_rooms',
    'name', 'description', 'neighborhood_overview', 'neighbourhood',
    'property_type',
    'has_availability', 'availability_30', 'availability_60',
    'availability_90', 'availability_365', 'availability_eoy',
    
    
    'first_review', 'last_review',
    
    'license', 'bathrooms_text', 'bathrooms', 'amenities',
    'estimated_occupancy_l365d',
    'estimated_revenue_l365d',
    
     'number_of_reviews',
    'number_of_reviews_ltm',
    'number_of_reviews_l30d',
    'number_of_reviews_ly',
    'reviews_per_month',
    'verifications_count'
]

df_listings.drop(columns=cols_drop, inplace=True, errors='ignore')

In [84]:
df_listings.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1055 entries, 0 to 1357
Data columns (total 42 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              1055 non-null   int64  
 1   host_is_superhost               1055 non-null   int64  
 2   host_listings_count             1055 non-null   int64  
 3   host_total_listings_count       1055 non-null   int64  
 4   neighbourhood_cleansed          1055 non-null   object 
 5   latitude                        1055 non-null   float64
 6   longitude                       1055 non-null   float64
 7   accommodates                    1055 non-null   int64  
 8   bedrooms                        1055 non-null   float64
 9   beds                            1055 non-null   float64
 10  price                           1055 non-null   float64
 11  minimum_nights                  1055 non-null   int64  
 12  maximum_nights                  1055 no

In [85]:
df_listings.shape

(1055, 42)

In [ ]:
df_listings.to_csv('../data/processed/df_listings_cleaned.csv', index=False)